# ***1. Carga de librerias***

In [300]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import os

sys.path.append(os.path.abspath(".."))


# ***2. Carga del dataset***

In [301]:
df_pois = pd.read_csv('../data/pois_barcelona_enriquecidos_con_valoracion.csv')
df_pois.head(3)

,Unnamed: 0,id,name,country_latitude,country_longitude,city,city_latitude,city_longitude,poi,poi_latitude,...,category,subcategory,value_en,opening_hours,opening_hours_source,rating,reviews_count,price_level,google_place_id,match_confidence
0,2,3,Spain,39.3261,-4.83798,Barcelona,41.382894,2.177432,Biblioteca de Cataluña,41.3810,...,Cultural,library,national library,"Mo-Fr 09:00-20:00+; Sa 09:00-14:00+; PH,Su off",OSM,4.2,NaN,NaN,NaN,0.93
1,33,34,Spain,39.3261,-4.83798,Barcelona,41.382894,2.177432,Esquerra de l'Eixample,41.3868,...,Cultural,library,neighborhood,NaN,NaN,4.1,NaN,NaN,NaN,NaN
2,34,35,Spain,39.3261,-4.83798,Barcelona,41.382894,2.177432,Can Mariner (Barcelona),41.4312,...,Cultural,library,masia,"Mo,Fr 15:30-20:00; Tu,Th 09:30-13:30,15:30-20:...",OSM,4.1,NaN,NaN,NaN,0.65


# ***3. Limpieza de registros, columnas, nulos y duplicados***

In [302]:
# Creación de una copia para no trabajar sobre el dataset original
df_pois_cleaning = df_pois.copy()
df_pois_cleaning.shape

(779, 21)

In [303]:
# Filtro para averiguar si hay algún POI fuera de España o con coordenadas distintas a la Barcelona española
df_pois_cleaning[
    (df_pois_cleaning["poi_latitude"] < 20) |
    (df_pois_cleaning["poi_longitude"] < 0)
]

,Unnamed: 0,id,name,country_latitude,country_longitude,city,city_latitude,city_longitude,poi,poi_latitude,...,category,subcategory,value_en,opening_hours,opening_hours_source,rating,reviews_count,price_level,google_place_id,match_confidence
425,274799,295690,Venezuela,8.00187,-66.1109,Barcelona,10.137301,-64.686253,Casa Fuerte de Barcelona,10.137,...,Monumento,monument,NaN,NaN,NaN,4.7,NaN,NaN,NaN,NaN


In [304]:
df_pois_cleaning = df_pois_cleaning[
    df_pois_cleaning["name"].str.lower().str.strip() != "venezuela"
]
df_pois_cleaning.shape

(778, 21)

En el apartado anterior he excluido el POI correspondiente a la ciudad de Barcelona ubicada en Venezuela. He realizado este primer paso ya que al contemplar la opcion de eliminar la columna correspondiente al nombre del pais en pasos posteriores, para asi no dejar este registro junto a los POIs correctos.

## ***3.1 Eliminación de columnas innecesarias***

In [305]:
# Selección de las columnas que queremos borrar
cols_drop = [
    "Unnamed: 0",
    "name",
    "country_latitude",
    "country_longitude",
    "city_latitude",
    "city_longitude",
    "google_place_id",
    "price_level",
    "reviews_count"
]

# Eliminación de las columnas
df_pois_cleaning = df_pois_cleaning.drop(columns=cols_drop, errors="ignore")
df_pois_cleaning.head(1)

,id,city,poi,poi_latitude,poi_longitude,category,subcategory,value_en,opening_hours,opening_hours_source,rating,match_confidence
0,3,Barcelona,Biblioteca de Cataluña,41.381,2.16951,Cultural,library,national library,"Mo-Fr 09:00-20:00+; Sa 09:00-14:00+; PH,Su off",OSM,4.2,0.93


## ***3.2 Normalización del texto***

In [306]:
df_pois_cleaning["category"] = df_pois_cleaning["category"].str.strip().str.lower()
df_pois_cleaning["subcategory"] = df_pois_cleaning["subcategory"].str.strip().str.lower()
df_pois_cleaning["poi"] = df_pois_cleaning["poi"].str.strip()

## ***3.3 Eliminación de duplicados***

In [307]:
df_pois_cleaning = df_pois_cleaning.drop_duplicates(subset=["poi", "poi_latitude", "poi_longitude", "subcategory"])

In [308]:
df_pois_cleaning.shape

(775, 12)

## ***3.4 Normalización de columnas***

En el siguiente apartado, se normalizarán y cambiarán los nombres de las columnas.

In [309]:
from functions.normalize import normalize_column_names

df_pois_cleaning = normalize_column_names(df_pois_cleaning)

rename_dict = {
    "poi": "name",
    "value_en": "description",
    "poi_latitude": "latitude",
    "poi_longitude": "longitude"
}

df_pois_cleaning = df_pois_cleaning.rename(columns=rename_dict)

El siguiente paso busca reordenar las columnas para que sea más sencilla y más clara su lectura.

In [310]:
# Ordenación de las columnas
df_pois_cleaning = df_pois_cleaning[
    [
        "id",
        "name",
        "category",
        "subcategory",
        "latitude",
        "longitude",
        "city",
        "description",
        "rating",
        "match_confidence",
        "opening_hours",
        "opening_hours_source"
        
    ]
]

In [311]:
df_pois_cleaning.head(2)

,id,name,category,subcategory,latitude,longitude,city,description,rating,match_confidence,opening_hours,opening_hours_source
0,3,Biblioteca de Cataluña,cultural,library,41.3810,2.16951,Barcelona,national library,4.2,0.93,"Mo-Fr 09:00-20:00+; Sa 09:00-14:00+; PH,Su off",OSM
1,34,Esquerra de l'Eixample,cultural,library,41.3868,2.15205,Barcelona,neighborhood,4.1,NaN,NaN,NaN


## ***3.5 Creación y transformación de columnas de horario para modelado posterior***

In [312]:
from functions.preprocessing import normalize_opening_hours

# Utilizamos la funcion que tenemos en un archivo externo para sacar datos de la columna "opening_hours"
tmp = df_pois_cleaning["opening_hours"] \
    .apply(normalize_opening_hours) \
    .apply(pd.Series)

# Esto aplica para cada registro
df_pois_cleaning = pd.concat([df_pois_cleaning, tmp], axis=1)

In [313]:
# Muestro las columnas para comprobar que ha funcionado
df_pois_cleaning.columns

Index(['id', 'name', 'category', 'subcategory', 'latitude', 'longitude',
       'city', 'description', 'rating', 'match_confidence', 'opening_hours',
       'opening_hours_source', 'oh_norm', 'oh_is_null', 'oh_is_24_7',
       'oh_format'],
      dtype='object')

In [314]:
from functions.preprocessing import is_likely_open
# Crear features útiles desde las columnas oh_*
df_pois_cleaning["has_opening_hours"] = ~df_pois_cleaning["oh_is_null"]
df_pois_cleaning["is_24_7"] = df_pois_cleaning["oh_is_24_7"]

df_pois_cleaning["is_likely_open"] = df_pois_cleaning["opening_hours"].apply(is_likely_open)

# Eliminar columnas técnicas (ya no hacen falta)
df_pois_cleaning = df_pois_cleaning.drop(
    columns=["oh_norm", "oh_is_null", "oh_is_24_7", "oh_format"],
    errors="ignore"
)

In [315]:
df_pois_cleaning.head()

,id,name,category,subcategory,latitude,longitude,city,description,rating,match_confidence,opening_hours,opening_hours_source,has_opening_hours,is_24_7,is_likely_open
0,3,Biblioteca de Cataluña,cultural,library,41.3810,2.16951,Barcelona,national library,4.2,0.93,"Mo-Fr 09:00-20:00+; Sa 09:00-14:00+; PH,Su off",OSM,True,False,True
1,34,Esquerra de l'Eixample,cultural,library,41.3868,2.15205,Barcelona,neighborhood,4.1,NaN,NaN,NaN,False,False,False
2,35,Can Mariner (Barcelona),cultural,library,41.4312,2.16056,Barcelona,masia,4.1,0.65,"Mo,Fr 15:30-20:00; Tu,Th 09:30-13:30,15:30-20:...",OSM,True,False,True
3,111,Archivo Histórico de la Ciudad de Barcelona,cultural,library,41.3841,2.17567,Barcelona,archive,4.3,NaN,NaN,NaN,False,False,False
4,112,Arxiu Diocesà de Barcelona,cultural,library,41.3837,2.17534,Barcelona,episcopal archive,4.2,0.75,Mo off; Tu-We 18:00-01:00; Th 18:00-02:00; Fr ...,OSM,True,False,True


## ***3.6 Arreglo de horarios nulos***

In [316]:
# Se le atribuye que has_match_confidence = True a aquellos registros que el match_confidence no es nulo
df_pois_cleaning["has_match_confidence"] = df_pois_cleaning["match_confidence"].notna()

# Le imputo una media a aquellos registros que lo tengan nulo para no perder datos
mean_mc = df_pois_cleaning["match_confidence"].mean()
df_pois_cleaning["match_confidence"] = df_pois_cleaning["match_confidence"].fillna(mean_mc)

En el siguiente apartado creo una columna nueva correspondiente a una puntuación para los POIs basados en el match confidence. Si "match_confidence" es NaN, el score del POI sufrirá una penalización. Esto viene bien para la recomendación de rutas en fases más avanzadas del proyecto.

In [317]:
df_pois_cleaning["score"] = (
    df_pois_cleaning["rating"] * 0.7 +
    df_pois_cleaning["match_confidence"] * 0.3
)

# Penalización
df_pois_cleaning.loc[
    df_pois_cleaning["has_match_confidence"] == False, "score"
] *= 0.85

In [318]:
df_pois_cleaning.head()

,id,name,category,subcategory,latitude,longitude,city,description,rating,match_confidence,opening_hours,opening_hours_source,has_opening_hours,is_24_7,is_likely_open,has_match_confidence,score
0,3,Biblioteca de Cataluña,cultural,library,41.3810,2.16951,Barcelona,national library,4.2,0.930000,"Mo-Fr 09:00-20:00+; Sa 09:00-14:00+; PH,Su off",OSM,True,False,True,True,3.21900
1,34,Esquerra de l'Eixample,cultural,library,41.3868,2.15205,Barcelona,neighborhood,4.1,0.503766,NaN,NaN,False,False,False,False,2.56796
2,35,Can Mariner (Barcelona),cultural,library,41.4312,2.16056,Barcelona,masia,4.1,0.650000,"Mo,Fr 15:30-20:00; Tu,Th 09:30-13:30,15:30-20:...",OSM,True,False,True,True,3.06500
3,111,Archivo Histórico de la Ciudad de Barcelona,cultural,library,41.3841,2.17567,Barcelona,archive,4.3,0.503766,NaN,NaN,False,False,False,False,2.68696
4,112,Arxiu Diocesà de Barcelona,cultural,library,41.3837,2.17534,Barcelona,episcopal archive,4.2,0.750000,Mo off; Tu-We 18:00-01:00; Th 18:00-02:00; Fr ...,OSM,True,False,True,True,3.16500


Se optó por una estrategia combinada para el tratamiento de valores nulos en 
*match_confidence*, consistente en la imputación mediante la media y la 
incorporación de una penalización en función de la calidad del dato, permitiendo 
así mantener la información sin introducir sesgos significativos.

## ***3.7 Normalización de los valores de categoria de Español a Inglés***

In [319]:
# Veo los valores únicos de category
df_pois_cleaning["category"].unique()

array(['cultural', 'parque', 'fuente', 'punto de interés', 'histórico',
       'monumento', 'puente', 'religioso', 'administrativo', nan],
      dtype=object)

In [320]:
# Creación de un diccionario para cambiar el idioma de los valores
category_map = {
    "cultural": "cultural",
    "parque": "park",
    "fuente": "fountain",
    "punto de interés": "tourist_attraction",
    "histórico": "historic",
    "monumento": "monument",
    "puente": "bridge",
    "religioso": "religious",
    "administrativo": "administrative"
}

# Limpio los valores y los mapeo
df_pois_cleaning["category"] = (
    df_pois_cleaning["category"]
    .str.lower()
    .str.strip()
    .map(category_map)
)

# Arreglo los Nan imputandolos como 'Unkown'
df_pois_cleaning["category"] = df_pois_cleaning["category"].fillna("unknown")

In [321]:
# Comprobación de que se han cambiado correctamente
df_pois_cleaning["category"].unique()

array(['cultural', 'park', 'fountain', 'tourist_attraction', 'historic',
       'monument', 'bridge', 'religious', 'administrative', 'unknown'],
      dtype=object)

Se realizó una estandarización de la variable *category*, unificando los valores 
en inglés y eliminando inconsistencias derivadas del uso de distintos idiomas 
y formatos, con el objetivo de mejorar la coherencia del dataset.

# ***4. Parseo de tipos de las columnas***

In [322]:
# Como la columna category ya está normalizada, solo queda cambiarle el tipo
df_pois_cleaning["category"] = df_pois_cleaning["category"].astype("category")

In [323]:
# Limpieza de la columna de subcategory y parseo
df_pois_cleaning["subcategory"] = (
    df_pois_cleaning["subcategory"]
    .str.lower()
    .str.strip()
    .astype("category")
)
df_pois_cleaning["subcategory"] = df_pois_cleaning["subcategory"].astype("category")

In [324]:
# Seleccionamos las columnas que queremos que sean booleanos
cols_bool = [
    "has_opening_hours",
    "is_24_7",
    "is_likely_open",
    "has_match_confidence"
]

# Bucle para el parseo
for col in cols_bool:
    df_pois_cleaning[col] = df_pois_cleaning[col].astype(bool)

# Parseo de las columnas numéricas
df_pois_cleaning["rating"] = df_pois_cleaning["rating"].astype(float)
df_pois_cleaning["match_confidence"] = df_pois_cleaning["match_confidence"].astype(float)
df_pois_cleaning["score"] = df_pois_cleaning["score"].astype(float)
df_pois_cleaning["name"] = df_pois_cleaning["name"].astype(str)

df_pois_cleaning["has_valid_source"] = df_pois_cleaning["opening_hours_source"].notna()

Después de haber parseado todas las columnas a sus tipos correspondientes, comprobamos que todo haya salido bien.

In [325]:
df_pois_cleaning.dtypes

id                         int64
name                      object
category                category
subcategory             category
latitude                 float64
longitude                float64
city                      object
description               object
rating                   float64
match_confidence         float64
opening_hours             object
opening_hours_source      object
has_opening_hours           bool
is_24_7                     bool
is_likely_open              bool
has_match_confidence        bool
score                    float64
has_valid_source            bool
dtype: object

# ***5. Creación de columnas para la duración de visita de los POIs en base a su subcategoria***

A continuación, le atribuyo unas duraciones estimadas en minutos a las subcategorias de los POIs para que luego en el modelo de recomendación nos podamos basar en estos datos para que sea más eficiente e interactivo con el usuario.

In [326]:
# Duraciones estimadas en minutos (puedes ajustar fácilmente)
visit_duration_map = {
    # Cultura
    "museum": 120,
    "library": 60,
    "gallery": 60,
    "arts_centre": 75,
    "conservatory": 60,
    "theatre": 120,

    # Arte puntual
    "artwork": 15,

    # Histórico
    "monument": 25,
    "memorial": 20,
    "ruins": 45,
    "castle": 90,
    "archaeological_site": 60,
    "bunker": 20,
    "city_gate": 15,

    # Religioso
    "church": 40,
    "chapel": 25,
    "cathedral": 60,
    "monastery": 60,
    "synagogue": 40,

    # Urbano / rápido
    "fountain": 10,
    "bridge": 10,
    "square": 20,

    # Institucional
    "embassy": 20,
    "townhall": 30,
    "government": 20,
    "courthouse": 20,

    # General
    "attraction": 60
}

# Default razonable si no está en el mapa
DEFAULT_VISIT_DURATION = 45

df_pois_cleaning["visit_duration"] = (
    df_pois_cleaning["subcategory"].astype(str).str.lower().str.strip()
    .map(visit_duration_map)
    .fillna(DEFAULT_VISIT_DURATION)
    .astype(int)
)

In [327]:
df_pois_cleaning.head()

,id,name,category,subcategory,latitude,longitude,city,description,rating,match_confidence,opening_hours,opening_hours_source,has_opening_hours,is_24_7,is_likely_open,has_match_confidence,score,has_valid_source,visit_duration
0,3,Biblioteca de Cataluña,cultural,library,41.3810,2.16951,Barcelona,national library,4.2,0.930000,"Mo-Fr 09:00-20:00+; Sa 09:00-14:00+; PH,Su off",OSM,True,False,True,True,3.21900,True,60
1,34,Esquerra de l'Eixample,cultural,library,41.3868,2.15205,Barcelona,neighborhood,4.1,0.503766,NaN,NaN,False,False,False,False,2.56796,False,60
2,35,Can Mariner (Barcelona),cultural,library,41.4312,2.16056,Barcelona,masia,4.1,0.650000,"Mo,Fr 15:30-20:00; Tu,Th 09:30-13:30,15:30-20:...",OSM,True,False,True,True,3.06500,True,60
3,111,Archivo Histórico de la Ciudad de Barcelona,cultural,library,41.3841,2.17567,Barcelona,archive,4.3,0.503766,NaN,NaN,False,False,False,False,2.68696,False,60
4,112,Arxiu Diocesà de Barcelona,cultural,library,41.3837,2.17534,Barcelona,episcopal archive,4.2,0.750000,Mo off; Tu-We 18:00-01:00; Th 18:00-02:00; Fr ...,OSM,True,False,True,True,3.16500,True,60


# ***6. Creación de la comuna "tags" con función externa***

En este apartado, la finalidad es crear una columna "tags" para cada registro del dataset donde se le atribuya unas etiquetas en base a las categorias y subcategorias de los POIs.

Esto permitirá que el modelo de ML futuro pueda recomendar y aprender de mejor manera los POIs y la creación de las rutas.

In [328]:
from functions.tags import build_tags
df_pois_cleaning["tags"] = df_pois_cleaning.apply(build_tags, axis=1)

In [329]:
df_pois_cleaning.head(3)

,id,name,category,subcategory,latitude,longitude,city,description,rating,match_confidence,opening_hours,opening_hours_source,has_opening_hours,is_24_7,is_likely_open,has_match_confidence,score,has_valid_source,visit_duration,tags
0,3,Biblioteca de Cataluña,cultural,library,41.3810,2.16951,Barcelona,national library,4.2,0.930000,"Mo-Fr 09:00-20:00+; Sa 09:00-14:00+; PH,Su off",OSM,True,False,True,True,3.21900,True,60,"[culture, has_schedule, indoor]"
1,34,Esquerra de l'Eixample,cultural,library,41.3868,2.15205,Barcelona,neighborhood,4.1,0.503766,NaN,NaN,False,False,False,False,2.56796,False,60,"[culture, indoor]"
2,35,Can Mariner (Barcelona),cultural,library,41.4312,2.16056,Barcelona,masia,4.1,0.650000,"Mo,Fr 15:30-20:00; Tu,Th 09:30-13:30,15:30-20:...",OSM,True,False,True,True,3.06500,True,60,"[culture, has_schedule, indoor]"


El siguiente código crea una columna extra de tags para guardar como string las etiquetas.

Esto se debe a que al exportar el dataset a csv, puede dar problemas o fallos con las listas normales, asi que además de eso decido guardarlas como strings separados por |.

In [330]:
# Para cada registro creamos un string con las tags
df_pois_cleaning["tags_str"] = df_pois_cleaning["tags"].apply(lambda x: "|".join(x))
df_pois_cleaning.head(3)

,id,name,category,subcategory,latitude,longitude,city,description,rating,match_confidence,...,opening_hours_source,has_opening_hours,is_24_7,is_likely_open,has_match_confidence,score,has_valid_source,visit_duration,tags,tags_str
0,3,Biblioteca de Cataluña,cultural,library,41.3810,2.16951,Barcelona,national library,4.2,0.930000,...,OSM,True,False,True,True,3.21900,True,60,"[culture, has_schedule, indoor]",culture|has_schedule|indoor
1,34,Esquerra de l'Eixample,cultural,library,41.3868,2.15205,Barcelona,neighborhood,4.1,0.503766,...,NaN,False,False,False,False,2.56796,False,60,"[culture, indoor]",culture|indoor
2,35,Can Mariner (Barcelona),cultural,library,41.4312,2.16056,Barcelona,masia,4.1,0.650000,...,OSM,True,False,True,True,3.06500,True,60,"[culture, has_schedule, indoor]",culture|has_schedule|indoor


### ***Comprobaciones finales de la creación de las columnas***

In [331]:
# Hacemos un describe() de la columna "visit_duration"
df_pois_cleaning["visit_duration"].describe()

count    775.000000
mean      53.690323
std       36.461648
min       10.000000
25%       20.000000
50%       60.000000
75%       60.000000
max      120.000000
Name: visit_duration, dtype: float64

In [332]:
# Muestra de los primero 10 registros y su columnas "tags"
df_pois_cleaning["tags"].head(10)

0    [culture, has_schedule, indoor]
1                  [culture, indoor]
2    [culture, has_schedule, indoor]
3                  [culture, indoor]
4    [culture, has_schedule, indoor]
5                  [culture, indoor]
6    [culture, has_schedule, indoor]
7                  [culture, indoor]
8    [culture, has_schedule, indoor]
9    [culture, has_schedule, indoor]
Name: tags, dtype: object

In [333]:
# Muestra aleatoria de 5 tags como string para comprobar que se haya guardado bien 
df_pois_cleaning["tags_str"].sample(5)

273    culture|has_schedule|indoor|long_visit
513               culture|has_schedule|indoor
539                            culture|indoor
568                            culture|indoor
552               culture|has_schedule|indoor
Name: tags_str, dtype: object

# ***7. Exportación del dataset a csv y a parquet para preservación de tipos de las columnas***

In [334]:
# En esta celda exportamos el dataset ya limpiado y tratado para seguir con la parte de visualización
df_pois_cleaning.to_csv("../data/pois_barcelona_procesados.csv", index=False)
df_pois_cleaning.to_parquet("../data/pois_barcelona_procesados.parquet", index=False)

## 🧹 Conclusiones del proceso de limpieza y preparación de datos

En este notebook se ha llevado a cabo un proceso completo de limpieza, transformación y enriquecimiento del dataset de puntos de interés (POIs) de Barcelona, con el objetivo de dejar los datos preparados para su uso en fases posteriores de visualización y modelado.

### 🔧 Procesos realizados

- **Eliminación de registros erróneos o inconsistentes**, como POIs fuera de la localización objetivo.
- **Eliminación de duplicados** basados en coordenadas y nombre del POI.
- **Normalización de variables categóricas**, unificando idioma y formato (uso de minúsculas, eliminación de espacios).
- **Conversión de tipos de datos**, asegurando el uso correcto de variables categóricas, booleanas y numéricas.
- **Tratamiento de valores nulos**, incluyendo imputación en variables como `match_confidence` y creación de variables indicadoras.
- **Procesamiento de horarios (`opening_hours`)**, generando nuevas variables útiles como:
  - `has_opening_hours`
  - `is_24_7`
  - `is_likely_open`
- **Creación de variables derivadas clave para recomendación**:
  - `visit_duration`: duración estimada de visita en minutos según la subcategoría.
  - `tags`: conjunto de etiquetas que describen características del POI (tipo, entorno, duración, disponibilidad, etc.).
  - `tags_str`: representación en texto de las etiquetas para su uso en CSV o aplicaciones externas.
- **Creación de variables de calidad del dato**, como `has_match_confidence` y ajuste del `score`.

---

### 📦 Resultado

El dataset final se encuentra:

- Limpio y consistente
- Enriquecido con variables semánticas
- Preparado para sistemas de recomendación y generación de rutas

Se ha exportado en los siguientes formatos:

- **CSV** → para interoperabilidad y uso en frontend o visualización
- **Parquet** (opcional) → para mantener tipos de datos y optimizar rendimiento en procesos de Machine Learning

---

### 📚 Librerías utilizadas

Además de las librerías base de análisis de datos, se ha requerido:

- `pandas` → manipulación y transformación de datos
- `numpy` → operaciones numéricas
- `re` → procesamiento de texto (horarios)

Para exportación en formato parquet:

- `pyarrow` → motor necesario para soportar este formato

---

### 🚀 Próximos pasos

El dataset generado servirá como base para:

- Análisis exploratorio avanzado y visualización
- Desarrollo de un sistema de recomendación basado en preferencias del usuario
- Generación de rutas optimizadas en función de distancia, tiempo y características de los POIs

---